In [1]:
#Import necessary libraries
import pandas as pd
import requests
import json

# Url to the JSON data containing competition information
competitions_url = "https://raw.githubusercontent.com/statsbomb/open-data/master/data/competitions.json"
# Read the JSON data from the URL into a pandas DataFrame
competitions = pd.read_json(competitions_url)
# Display the first 15 rows of the DataFrame with available tournments
print (competitions[['competition_id', 'season_id', 'competition_name', 'season_name']].head(15))

# Get the events from a specific game
match_id = 8658
events_url = f"https://raw.githubusercontent.com/hudl/open-data/refs/heads/master/data/events/{match_id}.json"


raw = requests.get(events_url).json()
events = pd.json_normalize(raw)

print(events.shape)

# Keep only the passes and extract the x and y coordinates from the location column
passes = events[events['type.name'] == 'Pass'].copy()
passes['x'] = passes["location"].apply(lambda x: x[0] if isinstance(x, list) else None)
passes['y'] = passes["location"].apply(lambda x: x[1] if isinstance(x, list) else None)
passes[['minute', 'player.name', 'x', 'y']].head(10)



print(passes.columns.tolist())

    competition_id  season_id        competition_name season_name
0                9        281           1. Bundesliga   2023/2024
1                9         27           1. Bundesliga   2015/2016
2             1267        107  African Cup of Nations        2023
3               16          4        Champions League   2018/2019
4               16          1        Champions League   2017/2018
5               16          2        Champions League   2016/2017
6               16         27        Champions League   2015/2016
7               16         26        Champions League   2014/2015
8               16         25        Champions League   2013/2014
9               16         24        Champions League   2012/2013
10              16         23        Champions League   2011/2012
11              16         22        Champions League   2010/2011
12              16         21        Champions League   2009/2010
13              16         41        Champions League   2008/2009
14        

In [2]:
# Tells Python to look at modules inn \src folder
import sys
sys.path.append('../src')

# Import files already created in the src folder
from etl.extract import get_competitions, get_matches, get_events
from etl.transform import get_passes, build_player_match_stats
from etl.load import create_tables, save_to_db

# Create tables (for the first time)
create_tables(reset=True)

# Bring events from a specific match
match_id = 8658
event = get_events(match_id)

# Calculate stats on a specific player
player_name = "Luka Modrić"
stats = build_player_match_stats(event, player_name, match_id)

# Save the passes to a database
save_to_db(stats, "Luka_Modrić_Match_Stats")

# Verify
stats

Tables created successfully.
Saved: 1 rows in table 'Luka_Modrić_Match_Stats'


,match_id,player_name,total_passes,total_progressive_passes,total_won_duels,total_succesfull_interceptions
0,8658,Luka Modrić,77,20,0,0


In [3]:
# Import needed libraries
import pandas as pd
from sqlalchemy import create_engine

# Create a connection to the SQLite database
Engine = create_engine('sqlite:///../data/processed/football_scouting.db')

# Check if the data was saved correctly by querying the database
check = pd.read_sql('SELECT * FROM Passes_match_8658 LIMIT 5', Engine)
check

,minute,player.name,x,y
0,0,Mario Mandžukić,61.0,41.0
1,0,Marcelo Brozović,49.0,41.0
2,0,Luka Modrić,43.0,56.0
3,0,Šime Vrsaljko,38.0,78.0
4,0,Danijel Subašić,9.0,49.0


In [4]:
# List of players that played this match
players = event['player.name'].dropna().unique()
print(players)

<StringArray>
[                            'Mario Mandžukić',
                            'Marcelo Brozović',
                              'Olivier Giroud',
                                 'Luka Modrić',
                               'Šime Vrsaljko',
                              'Blaise Matuidi',
                             'Danijel Subašić',
                              'Raphaël Varane',
                                'Ivan Perišić',
                                'Ivan Rakitić',
                                'Ivan Strinić',
                                  'Paul Pogba',
                                'Domagoj Vida',
                                'Dejan Lovren',
                             'Benjamin Pavard',
                               'N''Golo Kanté',
                           'Antoine Griezmann',
                        'Kylian Mbappé Lottin',
                                 'Hugo Lloris',
                          'Samuel Yves Umtiti',
                          

In [5]:
print(events["duel.outcome.name"].unique())
print(events["interception.outcome.name"].unique())

<StringArray>
[nan, 'Lost Out', 'Success In Play', 'Won', 'Success Out']
Length: 5, dtype: str
<StringArray>
[nan, 'Won', 'Success In Play', 'Lost Out', 'Lost In Play']
Length: 5, dtype: str


In [6]:
# Import needed libraries
import sys
sys.path.append('../src')
import pandas as pd

from etl.extract import get_matches, get_events
from etl.transform import build_all_players_stats
from etl.load import create_tables, save_to_db

# Recreate tables with new columns
create_tables(reset=True)

matches = get_matches(43, 3)    # WorldCup 2018

all_stats = []

for match_id in matches['match_id'].head(10):
    events = get_events(match_id)
    all_stats.append(build_all_players_stats(events, match_id))
    print("Match:", match_id, "is ready!")

league_df = pd.concat(all_stats, ignore_index=True)
save_to_db(league_df, "Match_Metrics")

print("Rows and Columns: ", league_df.shape)
print("Positions: ", league_df['position'].unique())
league_df.head(10)

Tables created successfully.


AttributeError: 'Series' object has no attribute 'cum'